In [ ]:
# 1. Crop hotstart
# 2. resample to 500m grid

In [113]:
import xarray as xr
import numpy as np

from scipy.interpolate import RegularGridInterpolator

In [98]:
hotstart_200m = xr.open_dataset('/export/lv1/user/jvandermolen/model_output/active_runs/boundaries/dws_200m_nwes/restart_201501_dws200m_bio.nc')
bathy_200m = xr.open_dataset('/export/lv9/user/cgiannopoulos/home/pre-processing/bathymetry_smoothing/topo_adjusted_dws_200m_2009.nc')

print("Hotstart 200m dimensions:", dict(hotstart_200m.dims))
print("Bathy 200m dimensions:", dict(bathy_200m.dims))

# Scale xax and yax by 200 and assign back
hotstart_200m = hotstart_200m.assign_coords(
    xax=hotstart_200m.xax * 200,
    yax=hotstart_200m.yax * 200,
)

print("Verifying scaled offsets:")
print(f"Hotstart xax (x200): {hotstart_200m.xax.values}")
print(f"Hotstart yax (x200): {hotstart_200m.yax.values}")


Hotstart 200m dimensions: {'xax': 820, 'yax': 496, 'zax': 31}
Bathy 200m dimensions: {'yc': 486, 'xc': 820, 'xx': 821, 'yx': 487}
Verifying scaled offsets:
Hotstart xax (x200): [     0.    200.    400.    600.    800.   1000.   1200.   1400.   1600.
   1800.   2000.   2200.   2400.   2600.   2800.   3000.   3200.   3400.
   3600.   3800.   4000.   4200.   4400.   4600.   4800.   5000.   5200.
   5400.   5600.   5800.   6000.   6200.   6400.   6600.   6800.   7000.
   7200.   7400.   7600.   7800.   8000.   8200.   8400.   8600.   8800.
   9000.   9200.   9400.   9600.   9800.  10000.  10200.  10400.  10600.
  10800.  11000.  11200.  11400.  11600.  11800.  12000.  12200.  12400.
  12600.  12800.  13000.  13200.  13400.  13600.  13800.  14000.  14200.
  14400.  14600.  14800.  15000.  15200.  15400.  15600.  15800.  16000.
  16200.  16400.  16600.  16800.  17000.  17200.  17400.  17600.  17800.
  18000.  18200.  18400.  18600.  18800.  19000.  19200.  19400.  19600.
  19800.  20000.  20

/tmp/ipykernel_95418/1712769170.py:2: FutureWarning: In a future version, xarray will not decode timedelta values based on the presence of a timedelta-like units attribute by default. Instead it will rely on the presence of a timedelta64 dtype attribute, which is now xarray's default way of encoding timedelta64 values. To continue decoding timedeltas based on the presence of a timedelta-like units attribute, users will need to explicitly opt-in by passing True or CFTimedeltaCoder(decode_via_units=True) to decode_timedelta. To silence this warning, set decode_timedelta to True, False, or a 'CFTimedeltaCoder' instance.
  bathy_200m = xr.open_dataset('/export/lv9/user/cgiannopoulos/home/pre-processing/bathymetry_smoothing/topo_adjusted_dws_200m_2009.nc')
/tmp/ipykernel_95418/1712769170.py:4: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension 

In [102]:
print("Hotstart 200m dimensions:", dict(hotstart_200m.dims))
print("Bathy 200m dimensions:", dict(bathy_200m.dims))

Hotstart 200m dimensions: {'zax': 31, 'yax': 496, 'xax': 820}
Bathy 200m dimensions: {'yc': 486, 'xc': 820, 'xx': 821, 'yx': 487}


/tmp/ipykernel_95418/3480680765.py:1: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  print("Hotstart 200m dimensions:", dict(hotstart_200m.dims))
/tmp/ipykernel_95418/3480680765.py:2: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  print("Bathy 200m dimensions:", dict(bathy_200m.dims))


In [103]:
# Verify the offset between hotstart and bathymetry grids
print("Verifying offsets:")
print(f"Hotstart xax[0]: {hotstart_200m.xax.values[0]}")
print(f"Bathy xc[0]: {bathy_200m.xc.values[0]}")
print(f"Hotstart yax[10]: {hotstart_200m.yax.values[10]}")
print(f"Bathy yc[0]: {bathy_200m.yc.values[0]}")

print(f"Hotstart yax[10]: {hotstart_200m.yax.values[0:11]}")
print(f"Bathy yc[0]: {bathy_200m.yc.values[0:11]}")

# Check if they match as expected
x_offset = 0
y_offset = -10
print(f"\nExpected x offset: {x_offset} (hotstart starts at same x as bathy)")
print(f"Expected y offset: {y_offset} (hotstart starts 10 cells before bathy in y)")


# Crop the hotstart to match bathymetry dimensions
# Bathymetry: xc=820, yc=486
# Hotstart: xax=820, yax=496
# Slice xax[0:820], yax[10:496] (10 + 486 = 496)
cropped_hotstart = hotstart_200m.isel(xax=slice(0, 820), yax=slice(10, 496))

print("\nCropped hotstart dimensions:", dict(cropped_hotstart.dims))

# Verify that coordinates now match
print("\nVerifying cropped coordinates match bathymetry:")
print(f"Cropped hotstart xax[0]: {cropped_hotstart.xax.values[0]}")
print(f"Bathy xc[0]: {bathy_200m.xc.values[0]}")
print(f"Cropped hotstart yax[0]: {cropped_hotstart.yax.values[0]}")
print(f"Bathy yc[0]: {bathy_200m.yc.values[0]}")

# Check if all spatial dimensions match
print(f"\nSpatial dimensions match: xax={cropped_hotstart.dims['xax']} == xc={bathy_200m.dims['xc']}, yax={cropped_hotstart.dims['yax']} == yc={bathy_200m.dims['yc']}")


Verifying offsets:
Hotstart xax[0]: 0.0
Bathy xc[0]: 0.0
Hotstart yax[10]: 0.0
Bathy yc[0]: 0.0
Hotstart yax[10]: [200. 200. 200. 200. 200. 200. 200. 200. 200. 200.   0.]
Bathy yc[0]: [   0.  200.  400.  600.  800. 1000. 1200. 1400. 1600. 1800. 2000.]

Expected x offset: 0 (hotstart starts at same x as bathy)
Expected y offset: -10 (hotstart starts 10 cells before bathy in y)

Cropped hotstart dimensions: {'zax': 31, 'yax': 486, 'xax': 820}

Verifying cropped coordinates match bathymetry:
Cropped hotstart xax[0]: 0.0
Bathy xc[0]: 0.0
Cropped hotstart yax[0]: 0.0
Bathy yc[0]: 0.0

Spatial dimensions match: xax=820 == xc=820, yax=486 == yc=486


/tmp/ipykernel_95418/3369298884.py:24: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  print("\nCropped hotstart dimensions:", dict(cropped_hotstart.dims))
/tmp/ipykernel_95418/3369298884.py:34: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  print(f"\nSpatial dimensions match: xax={cropped_hotstart.dims['xax']} == xc={bathy_200m.dims['xc']}, yax={cropped_hotstart.dims['yax']} == yc={bathy_200m.dims['yc']}")


In [104]:
# Save cropped hotstart
cropped_hotstart.to_netcdf('/export/lv9/user/qzhan/home/pre-processing/hotstart/restart_201501_dws200m_bio_cropped.nc')
print("Cropped hotstart saved as '/export/lv9/user/qzhan/home/pre-processing/hotstart/restart_201501_dws200m_bio_cropped.nc'")

Cropped hotstart saved as '/export/lv9/user/qzhan/home/pre-processing/hotstart/restart_201501_dws200m_bio_cropped.nc'


In [106]:
hotstart_200m = xr.open_dataset('/export/lv9/user/qzhan/home/pre-processing/hotstart/restart_201501_dws200m_bio_cropped.nc')
hydro_500m = xr.open_dataset('/export/lv9/user/cgiannopoulos/home/pre-processing/hotstart/restart_dws_500m_201412_hydro_10_layers.nc')
bathy_500m = xr.open_dataset('/export/lv9/user/cgiannopoulos/home/pre-processing/bathymetry_resampling/topo_dws_500m.nc')

# Print dimensions to confirm
print("Hotstart 200m dimensions:", dict(hotstart_200m.dims))
print("Hydro 500m dimensions:", dict(hydro_500m.dims))


# Get target coordinates for 500m grid
xc_500 = hydro_500m.xax.values
yc_500 = hydro_500m.yax.values

print(f"Target grid sizes: yc={len(yc_500)}, xc={len(xc_500)}")

# Print coordinate ranges for debugging
source_x = hotstart_200m.xax.values
source_y = hotstart_200m.yax.values
print(f"Source x range: {source_x.min():.2f} to {source_x.max():.2f}")
print(f"Source y range: {source_y.min():.2f} to {source_y.max():.2f}")
print(f"Target x range: {xc_500.min():.2f} to {xc_500.max():.2f}")
print(f"Target y range: {yc_500.min():.2f} to {yc_500.max():.2f}")

Hotstart 200m dimensions: {'zax': 31, 'yax': 486, 'xax': 820}
Hydro 500m dimensions: {'xax': 328, 'yax': 194, 'zax': 11}
Target grid sizes: yc=194, xc=328
Source x range: 0.00 to 163800.00
Source y range: 0.00 to 97000.00
Target x range: 0.00 to 163500.00
Target y range: 0.00 to 96500.00


/tmp/ipykernel_95418/3580016118.py:6: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  print("Hotstart 200m dimensions:", dict(hotstart_200m.dims))
/tmp/ipykernel_95418/3580016118.py:7: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  print("Hydro 500m dimensions:", dict(hydro_500m.dims))


In [107]:
# Create landmask from bathymetry missing values
# Assuming missing_value = -10 as in the 200m bathy
missing_value = bathy_500m.bathymetry.missing_value if 'missing_value' in bathy_500m.bathymetry.attrs else -10.0
water_mask = bathy_500m.bathymetry.values > missing_value  # Water where bathymetry > missing_value
print(f"Water cells: {np.sum(water_mask)} out of {water_mask.size}")


Water cells: 31728 out of 63632


In [108]:
# Create meshgrid for interpolation points
Y_grid, X_grid = np.meshgrid(yc_500, xc_500, indexing='ij')

# Create new dataset for 500m hotstart
hotstart_500m = xr.Dataset()

# Set coordinates (keep names xax, yax, zax)
hotstart_500m['xax'] = ('xax', xc_500)
hotstart_500m['yax'] = ('yax', yc_500)
hotstart_500m['zax'] = hotstart_200m.zax  # Copy zax unchanged

# Copy global attributes
hotstart_500m.attrs = hotstart_200m.attrs

In [ ]:
# Interpolate each variable only at water cells
for var_name in hotstart_200m.data_vars:
    var = hotstart_200m[var_name]
    print(f"Interpolating {var_name} with dims {var.dims}")

    if var.dims == ('yax', 'xax'):
        # 2D variable interpolation
        data = var.values
        interp_func = RegularGridInterpolator((source_y, source_x), data, method='linear', bounds_error=False, fill_value=None)
        # Only interpolate at water points
        water_indices = np.where(water_mask.ravel())[0]
        target_y = Y_grid.ravel()[water_indices]
        target_x = X_grid.ravel()[water_indices]
        target_points = np.column_stack((target_y, target_x))
        interp_values = interp_func(target_points)
        new_values = np.full((len(yc_500), len(xc_500)), 0.0)
        new_values.ravel()[water_indices] = interp_values
        hotstart_500m[var_name] = (('yax', 'xax'), new_values)

    elif var.dims == ('zax', 'yax', 'xax'):
        # 3D variable interpolation (interpolate each z-level)
        new_values = np.zeros((31, len(yc_500), len(xc_500)))
        for k in range(31):
            data_k = var.values[k, :, :]
            interp_func = RegularGridInterpolator((source_y, source_x), data_k, method='linear', bounds_error=False, fill_value=None)
            # Only interpolate at water points
            water_indices = np.where(water_mask.ravel())[0]
            target_y = Y_grid.ravel()[water_indices]
            target_x = X_grid.ravel()[water_indices]
            target_points = np.column_stack((target_y, target_x))
            interp_values = interp_func(target_points)
            new_values[k].ravel()[water_indices] = interp_values
        hotstart_500m[var_name] = (('zax', 'yax', 'xax'), new_values)

    else:
        # Copy variables with other dimensions unchanged (scalars, 1D, etc.)
        hotstart_500m[var_name] = var

print("Interpolation complete. New dimensions:", dict(hotstart_500m.dims))

# Check for any NaN values in the interpolated data
nan_count = 0
for var_name in hotstart_500m.data_vars:
    var = hotstart_500m[var_name]
    if np.any(np.isnan(var.values)):
        nan_count += np.sum(np.isnan(var.values))
        print(f"Warning: {var_name} contains {np.sum(np.isnan(var.values))} NaN values")

if nan_count == 0:
    print("No NaN values found in the interpolated data.")
else:
    print(f"Total NaN values: {nan_count}")



Interpolating loop with dims ()
Interpolating julianday with dims ()
Interpolating secondsofday with dims ()
Interpolating timestep with dims ()
Interpolating z with dims ('yax', 'xax')
Interpolating zo with dims ('yax', 'xax')
Interpolating U with dims ('yax', 'xax')
Interpolating SlUx with dims ('yax', 'xax')
Interpolating Slru with dims ('yax', 'xax')
Interpolating V with dims ('yax', 'xax')
Interpolating SlVx with dims ('yax', 'xax')
Interpolating Slrv with dims ('yax', 'xax')
Interpolating ssen with dims ('yax', 'xax')
Interpolating ssun with dims ('yax', 'xax')
Interpolating ssvn with dims ('yax', 'xax')
Interpolating sseo with dims ('yax', 'xax')
Interpolating ssuo with dims ('yax', 'xax')
Interpolating ssvo with dims ('yax', 'xax')
Interpolating Uint with dims ('yax', 'xax')
Interpolating Vint with dims ('yax', 'xax')
Interpolating Uadv with dims ('yax', 'xax')
Interpolating Vadv with dims ('yax', 'xax')
Interpolating uu with dims ('zax', 'yax', 'xax')
Interpolating vv with dim

/tmp/ipykernel_95418/1503218342.py:39: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  print("Interpolation complete. New dimensions:", dict(hotstart_500m.dims))


Total NaN values: 10293
Saved to '/export/lv9/user/qzhan/model_output/active_runs/dws_500m/dws_500m/26x20/2015/01/restart_dws500m_2015_01.nc'


In [112]:
# Save the resampled hotstart file
#hotstart_500m.to_netcdf('../restart_dws_500m_201412_hydro.nc')

hotstart_500m.to_netcdf('/export/lv9/user/qzhan/home/pre-processing/hotstart/restart_dws500m_2015_01.nc')

print("Saved to '/export/lv9/user/qzhan/home/pre-processing/hotstart/restart_dws500m_2015_01.nc'")



Saved to '/export/lv9/user/qzhan/home/pre-processing/hotstart/restart_dws500m_2015_01.nc'


In [128]:
nlayers = len(hotstart_500m.zax) - 1
print(f"Number of layers (nlayers): {nlayers}")

nlayers_new = 10
print(f"New number of layers (nlayers_new): {nlayers_new}")

layer_ratio = nlayers / nlayers_new
print(f"Layer ratio (nlayers / nlayers_new): {layer_ratio:.2f} \n has to be integer for proper interpolation")

Number of layers (nlayers): 30
New number of layers (nlayers_new): 10
Layer ratio (nlayers / nlayers_new): 3.00 
 has to be integer for proper interpolation


In [129]:
# Subsample in z-direction for 10 layers (11 levels)
# Original zax: 31 levels (0 to 30)
# For 10 layers: subsample every 3 levels -> 11 levels: 0,3,6,...,30
z_step = 3
new_zax_indices = np.arange(0, 31, z_step)  # [0,3,6,...,30]
print(f"Subsampling zax from {len(hotstart_500m.zax)} to {len(new_zax_indices)} levels")

# Create subsampled dataset
hotstart_500m_sub = hotstart_500m.isel(zax=new_zax_indices)

# Update zax coordinate to new values from 0 to 10
new_zax_values = np.arange(len(new_zax_indices), dtype=float)  # 0, 1, 2, ..., 10
hotstart_500m_sub = hotstart_500m_sub.assign_coords(zax=new_zax_values)

print("Z-subsampling complete. New dimensions:", dict(hotstart_500m_sub.dims))
print(f"New zax values: {new_zax_values}")

# Check for any NaN values in the subsampled data
nan_count = 0
for var_name in hotstart_500m_sub.data_vars:
    var = hotstart_500m_sub[var_name]
    if np.any(np.isnan(var.values)):
        nan_count += np.sum(np.isnan(var.values))
        print(f"Warning: {var_name} contains {np.sum(np.isnan(var.values))} NaN values")

if nan_count == 0:
    print("No NaN values found in the subsampled data.")
else:
    print(f"Total NaN values: {nan_count}")

# Save the resampled and subsampled hotstart file
'/export/lv9/user/qzhan/home/pre-processing/hotstart/restart_dws500m_2015_01.nc'
hotstart_500m_sub.to_netcdf('/export/lv9/user/qzhan/home/pre-processing/hotstart/restart_dws500m_201501_10layers.nc')
print("Saved to '/export/lv9/user/qzhan/home/pre-processing/hotstart/restart_dws500m_201501_10layers.nc'")

Subsampling zax from 31 to 11 levels


/tmp/ipykernel_95418/3433107077.py:15: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  print("Z-subsampling complete. New dimensions:", dict(hotstart_500m_sub.dims))


Z-subsampling complete. New dimensions: {'xax': 328, 'yax': 194, 'zax': 11}
New zax values: [ 0.  1.  2.  3.  4.  5.  6.  7.  8.  9. 10.]
Total NaN values: 10293
Saved to '/export/lv9/user/qzhan/home/pre-processing/hotstart/restart_dws500m_201501_10layers.nc'


In [130]:
# replace hydrological variable in the hotstart file by the last time step of the hydro file (to ensure consistency with the boundary conditions)

# hydro 500m file generated by
# ncmerge -v /export/lv9/user/cgiannopoulos/model_output/active_runs/dws_500m/dws_500m/26x20/2015_backup/12/dws_500m.3d.00??.nc ~/home/pre-processing/hotstart/dws_500m.3d.hydro.28122015.nc

# Load the hydro 500m file again to get the last time step

hydro_500m_122015 = xr.open_dataset('/export/lv9/user/qzhan/home/pre-processing/hotstart/dws_500m.3d.hydro.28122015.nc')
hotstart_500m = xr.open_dataset('/export/lv9/user/qzhan/home/pre-processing/hotstart/restart_dws500m_201501_10layers.nc')

# Get the last time step (assuming time is the first dimension)
last_time_step = hydro_500m_122015.isel(time=-1)
print("Last time step dimensions:", dict(last_time_step.dims))

# Replace the hydrological variables in hotstart_500m with the last time step from hydro_500m_122015
for var_name in hydro_500m_122015.data_vars:
    if var_name in hotstart_500m.data_vars and hydro_500m_122015[var_name].dims == hotstart_500m[var_name].dims:
        print(f"Replacing variable {var_name} in hotstart with last time step from hydro file")
        hotstart_500m[var_name] = (hotstart_500m[var_name].dims, last_time_step[var_name].values)
    else:
        print(f"Variable {var_name} not found in hotstart or dimensions do not match, skipping replacement")


Last time step dimensions: {'level': 10, 'yc': 194, 'xc': 328}
Variable rho not found in hotstart or dimensions do not match, skipping replacement
Variable temp not found in hotstart or dimensions do not match, skipping replacement
Variable salt not found in hotstart or dimensions do not match, skipping replacement
Variable tke not found in hotstart or dimensions do not match, skipping replacement
Variable nuh not found in hotstart or dimensions do not match, skipping replacement
Variable num not found in hotstart or dimensions do not match, skipping replacement
Variable hvn not found in hotstart or dimensions do not match, skipping replacement
Variable hun not found in hotstart or dimensions do not match, skipping replacement
Variable h not found in hotstart or dimensions do not match, skipping replacement
Variable w not found in hotstart or dimensions do not match, skipping replacement
Variable vv not found in hotstart or dimensions do not match, skipping replacement
Variable uu not 

/tmp/ipykernel_95418/1656363383.py:13: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  print("Last time step dimensions:", dict(last_time_step.dims))


In [131]:
print(hotstart_500m.data_vars)
# meta data of the ecological variable
# ncdump /export/lv1/user/jvandermolen/model_output/active_runs/boundaries/dws_200m_nwes/bdy_dws_200m_2_2015_01.3d.nc | more

print(hydro_500m_122015.data_vars)
# ncdump /export/lv9/user/qzhan/home/pre-processing/hotstart/dws_500m.3d.hydro.28122015.nc | more

# variable names are different between the two files, so no replacement is done for the ecological variables, only for the hydrological variables (temperature, salinity, u, v)
# which variable names to take? Maybe depending on the model version.


Data variables:
    loop          int32 4B ...
    julianday     int32 4B ...
    secondsofday  int32 4B ...
    timestep      float64 8B ...
    z             (yax, xax) float64 509kB ...
    zo            (yax, xax) float64 509kB ...
    U             (yax, xax) float64 509kB ...
    SlUx          (yax, xax) float64 509kB ...
    Slru          (yax, xax) float64 509kB ...
    V             (yax, xax) float64 509kB ...
    SlVx          (yax, xax) float64 509kB ...
    Slrv          (yax, xax) float64 509kB ...
    ssen          (yax, xax) float64 509kB ...
    ssun          (yax, xax) float64 509kB ...
    ssvn          (yax, xax) float64 509kB ...
    sseo          (yax, xax) float64 509kB ...
    ssuo          (yax, xax) float64 509kB ...
    ssvo          (yax, xax) float64 509kB ...
    Uint          (yax, xax) float64 509kB ...
    Vint          (yax, xax) float64 509kB ...
    Uadv          (yax, xax) float64 509kB ...
    Vadv          (yax, xax) float64 509kB ...
    uu      

In [132]:
# Map hydro variable names to the hotstart variable names when they differ.
hydro_to_hotstart_varnames = {
    'temp': 'T',
}

# Replace the hydrological variables in hotstart_500m with the last time step from hydro_500m_122015
for hydro_var_name in last_time_step.data_vars:
    hotstart_var_name = hydro_to_hotstart_varnames.get(hydro_var_name, hydro_var_name)

    if hotstart_var_name in hotstart_500m.data_vars and last_time_step[hydro_var_name].dims == hotstart_500m[hotstart_var_name].dims:
        print(
            f"Replacing hotstart variable {hotstart_var_name} with hydro variable {hydro_var_name} from the last time step"
        )
        hotstart_500m[hotstart_var_name] = (
            hotstart_500m[hotstart_var_name].dims,
            last_time_step[hydro_var_name].values,
        )
    else:
        print(
            f"Hydro variable {hydro_var_name} -> hotstart variable {hotstart_var_name} not found or dimensions do not match, skipping replacement"
        )

Hydro variable rho -> hotstart variable rho not found or dimensions do not match, skipping replacement
Hydro variable temp -> hotstart variable T not found or dimensions do not match, skipping replacement
Hydro variable salt -> hotstart variable salt not found or dimensions do not match, skipping replacement
Hydro variable tke -> hotstart variable tke not found or dimensions do not match, skipping replacement
Hydro variable nuh -> hotstart variable nuh not found or dimensions do not match, skipping replacement
Hydro variable num -> hotstart variable num not found or dimensions do not match, skipping replacement
Hydro variable hvn -> hotstart variable hvn not found or dimensions do not match, skipping replacement
Hydro variable hun -> hotstart variable hun not found or dimensions do not match, skipping replacement
Hydro variable h -> hotstart variable h not found or dimensions do not match, skipping replacement
Hydro variable w -> hotstart variable w not found or dimensions do not match